In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

Integrating the Google Generative AI (Mostly works for all the tasks) and Groq models (works for the speed-critical tasks) for the all the tasks.

My idea is to use the gorq for the main chain and google gen AI for all the other chains and us the conditional to switch between the models.

In [2]:
from langchain_groq import ChatGroq

groq = ChatGroq(model="llama-3.1-8b-instant")

e:\git_projects\Projects\AI Productivity Agent\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Main Chain 

In [3]:
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()


class MainRoute(BaseModel):
    agent: Literal[
        "writer",
        "scheduler",
        "document",
        "code",
        "chat"
    ]


router_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
    You are an intelligent router.
    Choose exactly ONE agent:
    - writer → emails, reports, content creation
    - scheduler → calendar, reminders, deadlines
    - document → summarization, extraction, analysis
    - code → programming, debugging, code generation
    - chat → general conversation

     Respond ONLY with the agent name.
     """
    ),
    ("human", "{input}")
])



router_chain = router_prompt | groq.with_structured_output(MainRoute)
router_run = RunnableLambda(lambda x: router_chain.invoke({"input": x}))

Child Chains Like
"writer"
"scheduler",
"document",
"code",
"chat"

In [4]:
def chat_chain(input: str) -> str:

    chat_prompt = ChatPromptTemplate.from_messages([
        ("system",
            """
            You are a helpful conversational assistant.
            Answer naturally and clearly.
            """
        ),
        ("human", "{input}")
    ])

    chain = chat_prompt | groq | parser
    return chain.invoke({"input": input})
def writer_chain(input : str) -> str:    

    writer_prompt = ChatPromptTemplate.from_messages([
        ("system",
            """
            You are a professional writer.
            
            You generate high-quality, clear, well-structured content.
            Adapt tone to the user's intent.
            Avoid unnecessary verbosity.
            """
        ),
        ("human", "{input}")
    ])

    writer_chain = writer_prompt | groq | parser
    return writer_chain.invoke({"input": input})
def scheduler_chain(input: str) -> str:

    scheduler_prompt = ChatPromptTemplate.from_messages([
        ("system",
            """
            You are a scheduling assistant.
            
            Understand dates, times, deadlines, and reminders.
            Be precise and unambiguous.
            """
        ),
        ("human", "{input}")
    ])

    chain = scheduler_prompt | groq | parser
    return chain.invoke({"input": input})
def document_chain(input: str) -> str:

    document_prompt = ChatPromptTemplate.from_messages([
        ("system",
            """
            You are a document analysis assistant.
            
            Summarize, extract key points, or analyze content clearly.
            Structure the output when possible.
            """
        ),
        ("human", "{input}")
    ])

    chain = document_prompt | groq | parser
    return chain.invoke({"input": input})
def code_chain(input: str) -> str:

    code_prompt = ChatPromptTemplate.from_messages([
        ("system",
            """
            You are a senior software engineer.
            
            Write correct, clean, and efficient code.
            Follow best practices.
            Explain briefly only if necessary.
            """
        ),
        ("human", "{input}")
    ])

    chain = code_prompt | groq | parser
    return chain.invoke({"input": input})


writer_chain_run     = RunnableLambda(lambda x: writer_chain(x["input"]))
scheduler_chain_run  = RunnableLambda(lambda x: scheduler_chain(x["input"]))
document_chain_run   = RunnableLambda(lambda x: document_chain(x["input"]))
code_chain_run       = RunnableLambda(lambda x: code_chain(x["input"]))
chat_chain_run       = RunnableLambda(lambda x: chat_chain(x["input"]))


### Let's combine the the chains 

In [5]:
from langchain_core.runnables import RunnableBranch, RunnablePassthrough
input_adapter = RunnableLambda(lambda x: {"input": x})


final_chain = (
    input_adapter
    | RunnablePassthrough.assign(
        route=lambda x: router_run.invoke(x["input"])
    )
    | RunnableBranch(
        (lambda x: x["route"].agent == "writer",     writer_chain_run),
        (lambda x: x["route"].agent == "scheduler",  scheduler_chain_run),
        (lambda x: x["route"].agent == "document",   document_chain_run),
        (lambda x: x["route"].agent == "code",       code_chain_run),
        chat_chain_run,  # fallback
    )
)


In [6]:
print(final_chain.invoke("write about the pen"))
print(final_chain.invoke("fix this python bug"))
print(final_chain.invoke("summarize this document"))
print(final_chain.invoke("what is the capital of france"))

**The Timeless Symbol of Creativity: The Pen**

In an era dominated by digital technology, the pen remains an enduring symbol of creativity, self-expression, and personal touch. A simple yet powerful tool, the pen has been an essential companion to writers, artists, and thinkers for centuries.

**History of the Pen**

The earliest known writing instruments date back to ancient civilizations, with the Egyptians and Greeks using reeds and brushes to write on papyrus and parchment. As civilizations evolved, so did the design of writing instruments, with the introduction of metal nibs and ink. The modern pen, however, is credited to the 19th century, with the invention of the fountain pen by Lewis Edson Waterman in 1884.

**Types of Pens**

Today, there are numerous types of pens available, each with its unique characteristics and uses. Some of the most popular types include:

1. **Ballpoint Pens**: These pens use a small ball to dispense ink onto the paper, making them quick and convenien

In [8]:
from typing import Literal, Dict, Callable
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnablePassthrough
from langchain_groq import ChatGroq

parser = StrOutputParser()

groq = ChatGroq(
    model="llama-3.1-8b-instant",  # free supported model
    temperature=0
)

class MainRoute(BaseModel):
    agent: Literal[
        "writer", 
        "document", 
        "code", 
        "chat"
        ]


def make_agent_chain(system_prompt: str) -> RunnableLambda:
    """
    Returns a RunnableLambda for a given system prompt.
    The agent expects a dict with key 'input'.
    """
    def _agent(input_text: str) -> str:
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            
            ("human", "{input}")
        ])
        return (prompt | groq | parser).invoke({"input": input_text})

    return RunnableLambda(lambda x: _agent(x["input"]))

AGENTS: Dict[str, RunnableLambda] = {
    "writer": make_agent_chain(
        """
        You are a professional writer.
        Generate high-quality, clear, and well-structured content.
        Adapt tone to the user's intent. Avoid unnecessary verbosity.
        """
    ),
    "document": make_agent_chain(
        """
        You are a document analysis assistant.
        Summarize, extract key points, or analyze content clearly.
        """
    ),
    "code": make_agent_chain(
        """
        You are a senior software engineer.
        Write correct, clean, and efficient code.
        Explain briefly only if necessary.
        """
    ),
    "chat": make_agent_chain(
        """
        You are a helpful conversational assistant.
        Answer naturally and clearly.
        """
    )
}

router_prompt = ChatPromptTemplate.from_messages([
    ("system",
    """
    You are an intelligent router.
    Choose exactly ONE agent:
    - writer → emails, reports, content creation
    - document → summarization, extraction, analysis
    - code → programming, debugging, code generation
    - chat → general conversation

    Respond ONLY with the agent name.
    """
    ),
    ("human", "{input}")
])

router_chain = router_prompt | groq.with_structured_output(MainRoute)
router_run = RunnableLambda(lambda x: router_chain.invoke({"input": x}))

def build_final_chain() -> RunnableLambda:
    """
    Returns the full router + conditional agent chain.
    Usage: final_chain.invoke("user input")
    """
    input_adapter = RunnableLambda(lambda x: {"input": x})
    return (
        input_adapter
        | RunnablePassthrough.assign(route=lambda x: router_run.invoke(x["input"]))
        | RunnableBranch(
            *( (lambda x, a=agent: x["route"].agent==agent, chain) for agent, chain in AGENTS.items() 
            ),
            AGENTS["chat"]
            
        )
    )

if __name__ == "__main__":
    final_chain = build_final_chain()
    print(final_chain.invoke("write about the pen"))
    print(final_chain.invoke("fix this python bug"))
    print(final_chain.invoke("summarize this document"))
    print(final_chain.invoke("what is the capital of france"))


The pen is a simple yet incredibly powerful tool that has been a cornerstone of human communication for centuries. Its history dates back to ancient civilizations, where early forms of writing instruments were made from reeds, sticks, and even animal bones.

The modern pen, however, is a relatively recent invention. The first metal nib pen was patented in 1828 by John Jacob Parker, an Englishman who improved upon earlier designs. This innovation allowed for more precise writing and paved the way for the development of modern pens.

Over time, pens have evolved to become an essential tool for writing, drawing, and even artistic expression. From the classic fountain pen to the modern ballpoint pen, each type of pen has its unique characteristics and advantages.

**Types of Pens:**

1. **Fountain Pen:** A classic writing instrument that uses ink flowing from a nib to create written lines. Fountain pens are known for their smooth writing experience and are often preferred by calligraphers 

In [ ]:
# Tool 1 [DuckDuckGo Search]
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

# Tool 2 [Wikipedia Search]
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaAPIWrapper()
wikipedia_tool = WikipediaQueryRun(api_wrapper=wikipedia)


# Tool 3 [Custom Tool]
from langchain.tools import tool

@tool
def custom_tool(arg: str) -> str:
    """ This is a custom tool used for weather forecast that returns the input string. """
    return "Sunny"



toolkit = [search_tool, wikipedia_tool, custom_tool]